In [1]:
import json
import openai
import pandas as pd
from sklearn.metrics import f1_score

,0,1
0,"And in that context, of course, they're liftin...",0
1,It's a number that is incredibly competitive i...,0
2,"Yeah, for the quarter, so it was a strong quar...",1
3,So the number would have been even better.,0
4,Of course we started in Music with Prime Music...,1


In [11]:
from openai import OpenAI
path = 'openai_key.txt'
with open(path) as f:
    for line in f.readlines():
        my_key=line

client = OpenAI(api_key = my_key)

In [ ]:
#prepare fine tune model
#build train and eval set with prompt1
'''
You are a NLP researcher. This is an argument unit mining task. You explain the reason and determine the given one sentence is claim or premise. If the sentence is a claim, you should return the class type 1. If the sentence is a premise, you should return the class type 0. Only return the number only
sentence:I mean, sometimes it's not that you came up with some brilliant strategy, it's just like really good work consistently over a long period of time.
Reason: The given sentence expresses an opinion or viewpoint about achieving success through consistent hard work over time, rather than relying on a single brilliant strategy. It presents a clear argument or stance on the topic, making it an argument unit.
output:1

sentence:So for example, we never participated as much in I'd call it non-developed markets, medium and small businesses with all of the sophisticated workloads.
Reason: The given sentence does not express a clear opinion or viewpoint, nor does it present an argument or stance on a particular issue. Instead, it seems to be stating a fact about the speaker's past participation in certain markets. Therefore, it does not meet the criteria for an argument unit
output: 0
'''
train_list=[]
for i in range(0, len(df)):
    sample={"messages": [{"role": "system", "content": '''You are a NLP researcher. This is an argument unit mining task. You explain the reason and determine the given one sentence is claim or premise. If the sentence is a claim, you should return the class type 1. If the sentence is a premise, you should return the class type 0. Only return the number only
sentence:I mean, sometimes it's not that you came up with some brilliant strategy, it's just like really good work consistently over a long period of time.
Reason: The given sentence expresses an opinion or viewpoint about achieving success through consistent hard work over time, rather than relying on a single brilliant strategy. It presents a clear argument or stance on the topic, making it an argument unit.
output:1

sentence:So for example, we never participated as much in I'd call it non-developed markets, medium and small businesses with all of the sophisticated workloads.
Reason: The given sentence does not express a clear opinion or viewpoint, nor does it present an argument or stance on a particular issue. Instead, it seems to be stating a fact about the speaker's past participation in certain markets. Therefore, it does not meet the criteria for an argument unit
output: 0'''} , 
                         {"role": "user", "content": 'sentence:'+str(df.loc[i][0])}, 
                         {"role": "assistant", "content": str(df.loc[i][1])}]}
 
    train_list.append(sample)

#open train and eval file
train = "train.jsonl"

# Open the file in write mode and write each dictionary as a JSON object on a separate line
with open(train, 'w') as file:
    for data in train_list:
        json.dump(data, file)
        file.write('\n') 

# %%load eval set
eval = pd.read_json('/home/zengwesley/NTUKaggle/team_dev.json')
eval_list=[]
for i in range(0, len(df)):
    sample={"messages": [{"role": "system", "content": '''You are a NLP researcher. This is an argument unit mining task. You explain the reason and determine the given one sentence is claim or premise. If the sentence is a claim, you should return the class type 1. If the sentence is a premise, you should return the class type 0. Only return the number only
sentence:I mean, sometimes it's not that you came up with some brilliant strategy, it's just like really good work consistently over a long period of time.
Reason: The given sentence expresses an opinion or viewpoint about achieving success through consistent hard work over time, rather than relying on a single brilliant strategy. It presents a clear argument or stance on the topic, making it an argument unit.
output:1

sentence:So for example, we never participated as much in I'd call it non-developed markets, medium and small businesses with all of the sophisticated workloads.
Reason: The given sentence does not express a clear opinion or viewpoint, nor does it present an argument or stance on a particular issue. Instead, it seems to be stating a fact about the speaker's past participation in certain markets. Therefore, it does not meet the criteria for an argument unit
output: 0'''} , 
                         {"role": "user", "content": 'sentence:'+str(df.loc[i][0])}, 
                         {"role": "assistant", "content": str(df.loc[i][1])}]}
 
    eval_list.append(sample)


# %% Save the eval set to a JSONL file
eval = "eval.jsonl"

# Open the file in write mode and write each dictionary as a JSON object on a separate line
#Use a txt file to read the api key
with open(eval, 'w') as file:
    for data in eval_list:
        json.dump(data, file)
        file.write('\n')  


# %% crate data upload to openai and get file id
train_file=client.files.create(
  file=open("train.jsonl", "rb"),
  purpose='fine-tune'
)
train_file # find the file id (FileObject(id='file-mX6ovaFhXt9wOhjLaRnwOyXs',)

# %%rate data upload to openai and get file id
eval_file=client.files.create(
  file=open("eval.jsonl", "rb"),
  purpose='fine-tune'
)
eval_file# find the file id (FileObject(id=another id,)

# %% fine tune with train and eval set
client.fine_tuning.jobs.create(
  training_file='file-mX6ovaFhXt9wOhjLaRnwOyXs', #train file id
  validation_file="file-7FoFNd2ItlqxlZX581ekecwz", #eval file id
  model="gpt-3.5-turbo-0125"
)

#load eval set



In [12]:
#first prompt inference with fine tune model
test = pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')

# Display the dataframe
test.head()

#prompt 1 inference with fine-tuned model

test_list=[]
for i in range(0, len(test)):
    completion = client.chat.completions.create(
        model="ft:gpt-3.5-turbo-0125:iis-academia-sinica::9WRzzG6t",
        messages=[{"role": "system", "content": '''You are a NLP researcher. This is an argument unit mining task. You explain the reason and determine the given one sentence is claim or premise. If the sentence is a claim, you should return the class type 1. If the sentence is a premise, you should return the class type 0. Only return the number only
sentence:I mean, sometimes it's not that you came up with some brilliant strategy, it's just like really good work consistently over a long period of time.
Reason: The given sentence expresses an opinion or viewpoint about achieving success through consistent hard work over time, rather than relying on a single brilliant strategy. It presents a clear argument or stance on the topic, making it an argument unit.
output:1

sentence:So for example, we never participated as much in I'd call it non-developed markets, medium and small businesses with all of the sophisticated workloads.
Reason: The given sentence does not express a clear opinion or viewpoint, nor does it present an argument or stance on a particular issue. Instead, it seems to be stating a fact about the speaker's past participation in certain markets. Therefore, it does not meet the criteria for an argument unit
output: 0'''}, 
                  {"role": "user", "content": 'sentence:'+str(test.loc[i][0])}]
    )
    response=completion.choices[0].message.content
    test_list.append(response)

#process the output into Kaggle format
test['predict']=test_list
test_submit=test[[0, 'predict']]
test_submit = test_submit.rename(columns={0: 'ID', 'predict': 'y_pred', 'label'=test[1]})

#save the output
test_submit.to_csv('prompt1_test.csv', index=False)

In [36]:
#some pred will have word, extract the number only
import re

def extract_first_number(string):
    match = re.search(r'\d+', string)
    if match:
        return int(match.group())
    else:
        return None

# Example usage
y_pred=[]
for i in range(len(data)):
    y_pred.append(extract_first_number(data['y_pred'][i]))

In [40]:
#calculate the f1 score for the prompt1 with fine tune model
gpt_result=y_pred
gt_result=data['label']
data=pd.read_csv('prompt1_test.csv')
micro_f1 = f1_score(gt_result, gpt_result, average='micro')
print(f'Micro F1 Score: {micro_f1}')

macro_f1 = f1_score(gt_result, gpt_result, average='macro')
print(f'Macro F1 Score: {macro_f1}')

weighted_f1 = f1_score(gt_result, gpt_result, average='weighted')
print(f'Weighted F1 Score: {weighted_f1}')

Micro F1 Score: 0.7358101135190919
Macro F1 Score: 0.4896021806552698
Weighted F1 Score: 0.7354724710446096


In [42]:
#second prompt inference with fine tune model
'''
You will be performing argument unit mining on a given sentence to determine if it is a claim or a premise. 
Carefully read the sentence and think about whether it expresses an opinion, viewpoint, or stance on an issue (making it a claim) or if it simply states a fact or supporting evidence (making it a premise).

Write out your reasoning for the classification inside reasoning tags.

Finally, if the sentence is a claim, output number 1 . If the sentence is a premise, output number 0 .
'''

test_list=[]
for i in range(0, len(test)):
    completion = client.chat.completions.create(
        model="ft:gpt-3.5-turbo-0125:iis-academia-sinica::9WRzzG6t",
        messages=[{"role": "system", "content": '''You will be performing argument unit mining on a given sentence to determine if it is a claim or a premise. 
Carefully read the sentence and think about whether it expresses an opinion, viewpoint, or stance on an issue (making it a claim) or if it simply states a fact or supporting evidence (making it a premise).

Write out your reasoning for the classification inside reasoning tags.

Finally, if the sentence is a claim, output number 1 . If the sentence is a premise, output number 0 .'''}, 
                  {"role": "user", "content": 'sentence:'+str(test.loc[i][0])}]
    )
    response=completion.choices[0].message.content
    test_list.append(response)

#process the output into Kaggle format
test['predict']=test_list
test_submit=test[[0, 'predict']]
test_submit = test_submit.rename(columns={0: 'ID', 'predict': 'y_pred', 'label':test[1]})

#save the output
test_submit.to_csv('prompt2_test.csv', index=False)

In [73]:
#some pred will have word, extract the number only
import re
data=pd.read_csv('prompt2_test.csv')
def extract_first_number(string):
    match = re.search(r'\d+', string)
    if match:
        return int(match.group())
    else:
        return 0

# Example usage
y_pred=[]
for i in range(len(data)):
    y_pred.append(extract_first_number(data['y_pred'][i]))
    #first_number = extract_first_number(string)
  # Output: 123

In [74]:
##second prompt inference  f1 score with fine tune model 
label=pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')
gpt_result=y_pred
gt_result=label[1]
micro_f1 = f1_score(gt_result, gpt_result, average='micro')
print(f'Micro F1 Score: {micro_f1}')

macro_f1 = f1_score(gt_result, gpt_result, average='macro')
print(f'Macro F1 Score: {macro_f1}')

weighted_f1 = f1_score(gt_result, gpt_result, average='weighted')
print(f'Weighted F1 Score: {weighted_f1}')

Micro F1 Score: 0.7254901960784313
Macro F1 Score: 0.7254875648595456
Weighted F1 Score: 0.725528787288757


In [75]:
#inference with prompt1 without fine tune model
test_list=[]
for i in range(0, len(test)):
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo-0125",
        messages=[{"role": "system", "content": '''You are a NLP researcher. This is an argument unit mining task. You explain the reason and determine the given one sentence is claim or premise. If the sentence is a claim, you should return the class type 1. If the sentence is a premise, you should return the class type 0. Only return the number only
sentence:I mean, sometimes it's not that you came up with some brilliant strategy, it's just like really good work consistently over a long period of time.
Reason: The given sentence expresses an opinion or viewpoint about achieving success through consistent hard work over time, rather than relying on a single brilliant strategy. It presents a clear argument or stance on the topic, making it an argument unit.
output:1

sentence:So for example, we never participated as much in I'd call it non-developed markets, medium and small businesses with all of the sophisticated workloads.
Reason: The given sentence does not express a clear opinion or viewpoint, nor does it present an argument or stance on a particular issue. Instead, it seems to be stating a fact about the speaker's past participation in certain markets. Therefore, it does not meet the criteria for an argument unit
output: 0'''}, 
                  {"role": "user", "content": 'sentence:'+str(test.loc[i][0])}]
    )
    response=completion.choices[0].message.content
    test_list.append(response)

#process the output into Kaggle format
test['predict']=test_list
test_submit=test[[0, 'predict']]
test_submit = test_submit.rename(columns={0: 'ID', 'predict': 'y_pred', 'label':label[1]})

#save the output
test_submit.to_csv('prompt1_wofinetune.csv', index=False)

In [76]:
#some pred will have word, extract the number only
import re
data=pd.read_csv('/home/zengwesley/NTUNLP/Task 2.1/prompt1_wofinetune.csv')
def extract_first_number(string):
    match = re.search(r'\d+', string)
    if match:
        return int(match.group())
    else:
        return 0

# Example usage
y_pred=[]
for i in range(len(data)):
    y_pred.append(extract_first_number(data['y_pred'][i]))
    #first_number = extract_first_number(string)
  # Output: 123

In [77]:
#second prompt inference wihtout fine tune model
label=pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')
gpt_result=y_pred
gt_result=label[1]
micro_f1 = f1_score(gt_result, gpt_result, average='micro')
print(f'Micro F1 Score: {micro_f1}')

macro_f1 = f1_score(gt_result, gpt_result, average='macro')
print(f'Macro F1 Score: {macro_f1}')

weighted_f1 = f1_score(gt_result, gpt_result, average='weighted')
print(f'Weighted F1 Score: {weighted_f1}')

Micro F1 Score: 0.5211558307533539
Macro F1 Score: 0.5161048689138577
Weighted F1 Score: 0.5185028002922044


In [78]:
#second prompt inference without fine tune model
'''
You will be performing argument unit mining on a given sentence to determine if it is a claim or a premise. 
Carefully read the sentence and think about whether it expresses an opinion, viewpoint, or stance on an issue (making it a claim) or if it simply states a fact or supporting evidence (making it a premise).

Write out your reasoning for the classification inside reasoning tags.

Finally, if the sentence is a claim, output number 1 . If the sentence is a premise, output number 0 .
'''

test_list=[]
for i in range(0, len(test)):
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo-0125",
        messages=[{"role": "system", "content": '''You will be performing argument unit mining on a given sentence to determine if it is a claim or a premise. 
Carefully read the sentence and think about whether it expresses an opinion, viewpoint, or stance on an issue (making it a claim) or if it simply states a fact or supporting evidence (making it a premise).

Write out your reasoning for the classification inside reasoning tags.

Finally, if the sentence is a claim, output number 1 . If the sentence is a premise, output number 0 .'''}, 
                  {"role": "user", "content": 'sentence:'+str(test.loc[i][0])}]
    )
    response=completion.choices[0].message.content
    test_list.append(response)

#process the output into Kaggle format
test['predict']=test_list
test_submit=test[[0, 'predict']]
test_submit = test_submit.rename(columns={0: 'ID', 'predict': 'y_pred', 'label':label[1]})

#save the output
test_submit.to_csv('prompt2_wofinetune.csv', index=False)

In [79]:
import re
data=pd.read_csv('/home/zengwesley/NTUNLP/Task 2.1/prompt2_wofinetune.csv')
def extract_first_number(string):
    match = re.search(r'\d+', string)
    if match:
        return int(match.group())
    else:
        return 0

# Example usage
y_pred=[]
for i in range(len(data)):
    y_pred.append(extract_first_number(data['y_pred'][i]))
    #first_number = extract_first_number(string)
  # Output: 123

In [80]:
label=pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')
gpt_result=y_pred
gt_result=label[1]
micro_f1 = f1_score(gt_result, gpt_result, average='micro')
print(f'Micro F1 Score: {micro_f1}')

macro_f1 = f1_score(gt_result, gpt_result, average='macro')
print(f'Macro F1 Score: {macro_f1}')

weighted_f1 = f1_score(gt_result, gpt_result, average='weighted')
print(f'Weighted F1 Score: {weighted_f1}')

Micro F1 Score: 0.608875128998968
Macro F1 Score: 0.04088219558235419
Weighted F1 Score: 0.6168191715234708
